In [1]:
import sys
sys.path.append("..")   # lets the notebook find your src/ folder

import pandas as pd
from src.preprocessing import load_heart, load_ckd, heart_preprocessor, ckd_preprocessor

In [2]:
X_heart, y_heart = load_heart("../data/raw/heart.csv")
X_ckd, y_ckd = load_ckd("../data/raw/ckd.csv")

print("Heart:", X_heart.shape, "| CKD:", X_ckd.shape)
print("Missing in heart:", X_heart.isnull().sum()[lambda s: s > 0].to_dict())
print("Missing in CKD (top 5):", X_ckd.isnull().sum().sort_values(ascending=False).head(5).to_dict())

Heart: (918, 11) | CKD: (400, 24)
Missing in heart: {'RestingBP': 1, 'Cholesterol': 172}
Missing in CKD (top 5): {'rbc': 152, 'rbcc': 131, 'wbcc': 106, 'pot': 88, 'sod': 87}


In [3]:
pre = heart_preprocessor()
clean = pre.fit_transform(X_heart)

print("Before:", X_heart.shape, "| After:", clean.shape)
print("Missing values after cleaning:", int(clean.isnull().sum().sum()))
clean.head(3).round(2)

Before: (918, 11) | After: (918, 18)
Missing values after cleaning: 0


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40.0,140.0,289.0,0.0,172.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,49.0,160.0,180.0,0.0,156.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,37.0,130.0,283.0,0.0,98.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def make_models(preprocessor_fn):
    return {
        "Logistic Regression": Pipeline([("clean", preprocessor_fn(scale=True)),
                                         ("model", LogisticRegression(max_iter=1000))]),
        "Random Forest": Pipeline([("clean", preprocessor_fn()),
                                   ("model", RandomForestClassifier(n_estimators=300, random_state=42))]),
        "XGBoost": Pipeline([("clean", preprocessor_fn()),
                             ("model", XGBClassifier(n_estimators=200, max_depth=3,
                                                     learning_rate=0.05, random_state=42))]),
    }

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

results = []
for dataset, X, y, fn in [("Heart", X_heart, y_heart, heart_preprocessor),
                          ("CKD", X_ckd, y_ckd, ckd_preprocessor)]:
    for name, pipe in make_models(fn).items():
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
        results.append({"Dataset": dataset, "Model": name,
                        "AUC (mean)": round(scores.mean(), 3),
                        "AUC (std)": round(scores.std(), 3)})

pd.DataFrame(results)

,Dataset,Model,AUC (mean),AUC (std)
0,Heart,Logistic Regression,0.922,0.018
1,Heart,Random Forest,0.924,0.022
2,Heart,XGBoost,0.927,0.021
3,CKD,Logistic Regression,1.000,0.000
4,CKD,Random Forest,1.000,0.000
5,CKD,XGBoost,0.999,0.001
